# 04 — Umbrella sampling and WHAM

**Learning goals**

- restrain the system in a series of windows along a reaction coordinate,
- see why neighbouring windows must overlap,
- combine biased windows into one unbiased profile with an iterative WHAM solver,
- recognise a poor reconstruction instead of trusting it.

In [ ]:
# Setup: make the `src` package importable (locally or on Google Colab).
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/HyeonjeYang/interactive-metadynamics.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/interactive-metadynamics")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    ROOT = Path.cwd().resolve()
    if not (ROOT / "src").is_dir():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import time as timer
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from src.potentials import double_well_force, double_well_landmarks, true_free_energy
from src.umbrella import run_umbrella_windows, umbrella_bias
from src.wham import wham_1d
from src.analysis import free_energy_rmse
from src.plotting import set_style, save_figure, plot_reference, free_energy_axes, REFERENCE, METHOD_COLORS
set_style()

## Parameters

`n_windows`, `spring_constant` and `window_steps` control overlap and statistics.

In [ ]:
a, b = 2.0, 0.5
T = 0.4
friction = 1.0
dt = 0.002

n_windows = 16
spring_constant = 40.0          # k_i, identical for all windows here
window_range = (-1.5, 1.4)      # first and last window centre
equil_steps = 2_000             # burn-in per window (discarded)
window_steps = 20_000           # production steps per window
sample_stride = 10

bins = np.linspace(-1.8, 1.7, 141)
tol, max_iter = 1e-8, 10_000    # WHAM convergence controls
seed = 2026

## 1. Restrained windows

Window $i$ adds a harmonic restraint
$$U_i(x) = \tfrac12 k_i(x-x_i)^2,$$
so it samples $P_i^\mathrm{b}(x) \propto P(x)\,e^{-\beta U_i(x)}$. Each window sees only a narrow slice of $x$,
of width roughly $\sqrt{k_BT/(k_i + V'')}$. To stitch the slices together, neighbouring windows must share samples.

In [ ]:
lm = double_well_landmarks(a, b)
force = lambda x: double_well_force(x, a, b)
centers = np.linspace(*window_range, n_windows)
print(f"window spacing = {centers[1] - centers[0]:.3f},  restraint width sqrt(kT/k) = {np.sqrt(T / spring_constant):.3f}")

## 2. Run all windows

Every window is an independent overdamped walker started at its centre: equilibrate, discard the burn-in, then
collect production samples. The windows never interact, so here they are integrated together as one NumPy array.

In [ ]:
def umbrella_wham(n_windows, spring_constant, window_steps, seed):
    """Run the windows and WHAM; return (windows, WHAM result, list of WHAM warnings)."""
    centers = np.linspace(*window_range, n_windows)
    run = run_umbrella_windows(centers, spring_constant, force, equil_steps, window_steps, dt, T,
                               np.random.default_rng(seed), friction, sample_stride)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        res = wham_1d(run["samples"], centers, spring_constant, T, bins, tol=tol, max_iter=max_iter)
    return run, res, [str(w.message) for w in caught]

t0 = timer.perf_counter()
windows, result, issues = umbrella_wham(n_windows, spring_constant, window_steps, seed)
print(f"{n_windows} windows x {equil_steps + window_steps} steps + WHAM: {timer.perf_counter() - t0:.1f} s")
print(f"samples per window: {windows['samples'].shape[0]}")

## 3. Window histograms and overlap

In [ ]:
ramp = LinearSegmentedColormap.from_list("windows", ["#86b6ef", "#0d366b"])
xc = result["x"]

def plot_windows(ax, res):
    n = len(res["centers"])
    for i, h in enumerate(res["counts"]):
        ax.fill_between(xc, h / h.sum() / np.diff(bins), color=ramp(i / max(n - 1, 1)), alpha=0.45, lw=0)
    ax.set_xlabel(r"$x$")
    ax.set_ylabel("biased density")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [2, 1]})
plot_windows(axes[0], result)
axes[0].set_title(f"{n_windows} windows, k = {spring_constant}", fontsize=10)
axes[1].bar(np.arange(len(result["overlaps"])), result["overlaps"], color=METHOD_COLORS["umbrella"])
axes[1].axhline(0.03, color=REFERENCE, ls="--", lw=1, label="warning threshold")
axes[1].set_xlabel("neighbouring pair (i, i+1)")
axes[1].set_ylabel("histogram overlap")
axes[1].legend(fontsize=8)
fig.tight_layout()
save_figure(fig, "04_umbrella_windows")
plt.show()

The overlap of two normalized histograms is $\sum_j\min(h_{ij}, h_{i+1,j})$ (1 = identical, 0 = disjoint).
The windows near the barrier are slightly wider because the restraint competes with the negative curvature there.

## 4. WHAM

With bin counts $n_{ij}$, $N_i$ samples in window $i$ and dimensionless window offsets $f_i$, iterate

$$p_j = \frac{\sum_i n_{ij}}{\sum_i N_i\,e^{f_i-\beta U_i(x_j)}}, \qquad
f_i = -\ln\sum_j p_j\,e^{-\beta U_i(x_j)}$$

until the offsets stop changing. `src/wham.py` evaluates both lines with `scipy.special.logsumexp`, fixes the
gauge $f_0=0$, leaves empty bins as NaN, and warns about non-convergence, poor overlap or gaps.

In [ ]:
print(f"converged: {result['converged']} after {result['n_iter']} iterations (tol = {tol})")
print("window offsets kT * f_i:", np.round(result["offsets"], 2))
print("WHAM issues:", issues if issues else "none")

F_ref = true_free_energy(xc, a, b)
region = F_ref <= lm["barrier_from_left"] + 2 * T
rmse, coverage = free_energy_rmse(result["free_energy"], F_ref, region)
print(f"RMSE vs F_true over the comparison region: {rmse / T:.2f} kT (coverage {coverage:.0%})")

fig, ax = plt.subplots(figsize=(6.5, 3.5))
plot_reference(ax, xc, F_ref)
ax.plot(xc, result["free_energy"], color=METHOD_COLORS["umbrella"], label="umbrella + WHAM")
free_energy_axes(ax, ylim=(-0.2, 4.5))
ax.legend()
fig.tight_layout()
plt.show()

Under these parameters the WHAM profile follows the known 1D reference across both basins and the barrier,
including the region that unbiased dynamics in Notebook 02 almost never visited.

## 5. Poor versus reasonable overlap

Same total idea, two settings: few stiff windows (gaps between them) versus the default.

In [ ]:
settings = {"poor: 6 windows, k = 200": (6, 200.0), f"reasonable: {n_windows} windows, k = {spring_constant:g}": (n_windows, spring_constant)}
fig, axes = plt.subplots(2, 2, figsize=(10, 5.6), sharex=True)
for col, (label, (nw, ks)) in enumerate(settings.items()):
    run, res, msgs = umbrella_wham(nw, ks, window_steps, seed)
    plot_windows(axes[0, col], res)
    axes[0, col].set_title(label, fontsize=10)
    plot_reference(axes[1, col], xc, F_ref)
    axes[1, col].plot(xc, res["free_energy"], color=METHOD_COLORS["umbrella"], label="WHAM")
    free_energy_axes(axes[1, col], ylim=(-0.2, 4.5))
    err = free_energy_rmse(res["free_energy"], F_ref, region)
    axes[1, col].set_title(f"RMSE {err[0] / T:.2f} kT on {err[1]:.0%} of region, reliable = {res['reliable']}", fontsize=9)
    print(f"{label}: converged = {res['converged']} ({res['n_iter']} iterations), min overlap = {res['overlaps'].min():.3f}")
    for m in msgs:
        print("   WARNING:", m)
axes[1, 0].legend()
fig.tight_layout()
plt.show()

With disjoint windows the WHAM equations decouple: *any* set of offsets is a fixed point, so the iteration
"converges" immediately to an arbitrary answer. Convergence of the solver is not evidence of a good result —
the overlap diagnostics and the warnings are.

## 6. Optional sweep

Set `RUN_SWEEP = True` to tabulate a few combinations (each takes well under a second).

In [ ]:
RUN_SWEEP = False
if RUN_SWEEP:
    print(f"{'windows':>8} {'k':>6} {'min overlap':>12} {'RMSE/kT':>8} {'reliable':>9}")
    for nw in (6, 10, 16):
        for ks in (20.0, 40.0, 100.0, 200.0):
            _, res, _ = umbrella_wham(nw, ks, window_steps, seed)
            err = free_energy_rmse(res["free_energy"], F_ref, region)[0]
            print(f"{nw:8d} {ks:6.0f} {res['overlaps'].min():12.3f} {err / T:8.2f} {str(res['reliable']):>9}")

## 7. A note on cost

Umbrella windows are independent and naturally parallel; here they are vectorized in one array, while a real
study would run them as separate jobs. Wall-clock times in these notebooks are illustrative, not benchmarks.

## Takeaways

- Umbrella sampling replaces one hard simulation by many easy, restrained ones along a chosen coordinate.
- WHAM determines the unknown window offsets self-consistently and returns one unbiased distribution.
- Overlap between neighbouring windows is what connects them; without it the result is undetermined.
- A converged WHAM iteration can still be wrong: check overlap, gaps and warnings before trusting a profile.

*References:* Torrie & Valleau, *J. Comput. Phys.* **23**, 187 (1977), doi:10.1016/0021-9991(77)90121-8 ·
Ferrenberg & Swendsen, *Phys. Rev. Lett.* **63**, 1195 (1989), doi:10.1103/PhysRevLett.63.1195 ·
Kumar et al., *J. Comput. Chem.* **13**, 1011 (1992), doi:10.1002/jcc.540130812.